# 🔐 LLM-Based Security Classification System
### NLP & Prompt Engineering | Google Gemini + HuggingFace Transformers

**Architecture:**
- 🤖 **Primary**: Google Gemini 1.5 Flash with structured prompt templates
- 📐 **Schema**: Pydantic output validation for JSON responses
- 🔍 **Quality Validator**: Confidence, consistency & completeness checks
- 🪂 **Fallback Degradation**: Gemini → Simplified Prompt → HuggingFace → Rule-based
- 📊 **Dataset**: [Malicious URLs Dataset](https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset) — 651,191 URLs across 4 threat classes

> ⚙️ **Setup**: Add your Gemini API key to Kaggle Secrets as `GEMINI_API_KEY`  
> Get a free key at: https://aistudio.google.com/apikey


In [ ]:
# Install required packages
!pip install google-generativeai pydantic transformers scikit-learn \
             matplotlib seaborn tqdm -q
print("✅ All packages installed!")


In [ ]:
import os, json, time, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from typing import Literal, List, Optional
from pydantic import BaseModel, Field
import google.generativeai as genai
from transformers import pipeline as hf_pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
print("✅ All imports successful!")


## 🔑 Step 1 — Gemini API Configuration

In [ ]:
# Load API key from Kaggle Secrets (add via: Notebook Settings > Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    print("✅ Loaded API key from Kaggle Secrets")
except Exception:
    # Fallback for local testing
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_KEY_HERE")
    print("⚠️  Using environment variable for API key")

genai.configure(api_key=GEMINI_API_KEY)

# Verify connection
try:
    test_model = genai.GenerativeModel('gemini-1.5-flash')
    test_resp  = test_model.generate_content("Reply with: OK")
    print(f"✅ Gemini API connected | Test: {test_resp.text.strip()}")
except Exception as e:
    print(f"❌ API Error: {e}")


## 📦 Step 2 — Load the Malicious URLs Dataset

**Dataset**: `sid321axn/malicious-urls-dataset` → `malicious_phish.csv`

| Column | Description |
|--------|-------------|
| `url`  | Raw URL string |
| `type` | Label: `benign`, `phishing`, `malware`, `defacement` |

> 🔧 **To add this dataset**: Notebook → Add Data → Search *"Malicious URLs"* by sid321axn


In [ ]:
# Load dataset
df = pd.read_csv("/kaggle/input/malicious-urls-dataset/malicious_phish.csv")
df.columns = df.columns.str.strip().str.lower()

print(f"📁 Total URLs: {len(df):,}")
print(f"\n📊 Class Distribution:")
print(df['type'].value_counts())

# Stratified sample — 50 per class (respects free-tier Gemini rate limits)
SAMPLE_PER_CLASS = 50
sample_df = (df.groupby('type')
               .apply(lambda x: x.sample(min(SAMPLE_PER_CLASS, len(x)), random_state=42))
               .reset_index(drop=True))

print(f"\n✅ Working sample: {len(sample_df)} URLs ({SAMPLE_PER_CLASS} per class)")
sample_df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Dataset Exploration — Malicious URLs", fontsize=13, fontweight='bold')

# Class distribution
counts = df['type'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title("Full Dataset — Class Distribution")
axes[0].set_ylabel("Count")
for i, (_, v) in enumerate(counts.items()):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=9)

# URL length distribution
df['url_length'] = df['url'].str.len()
for i, (cls, color) in enumerate(zip(['benign','phishing','malware','defacement'], colors)):
    subset = df[df['type'] == cls]['url_length']
    axes[1].hist(subset, bins=50, alpha=0.6, label=cls, color=color)
axes[1].set_title("URL Length Distribution by Class")
axes[1].set_xlabel("URL Length (chars)")
axes[1].set_ylabel("Frequency")
axes[1].set_xlim(0, 300)
axes[1].legend()

plt.tight_layout()
plt.show()


## 📐 Step 3 — Output Schemas (Pydantic)

In [ ]:
class SecurityClassification(BaseModel):
    """Structured output schema for LLM security classification."""
    classification: Literal["benign", "phishing", "malware", "defacement"]
    confidence:     float = Field(ge=0.0, le=1.0, description="Model confidence 0-1")
    risk_indicators: List[str] = Field(description="Observed threat signals")
    reasoning:      str  = Field(description="Short explanation of decision")
    severity:       Literal["low", "medium", "high", "critical"]


class ValidationResult(BaseModel):
    """Quality assessment result for a classification."""
    is_valid:        bool
    quality_score:   float = Field(ge=0.0, le=1.0)
    issues:          List[str]
    should_fallback: bool


print("✅ Pydantic schemas defined:")
print("  • SecurityClassification — 5 fields")
print("  • ValidationResult       — 4 fields")
print("\nExample schema:")
print(SecurityClassification.schema_json(indent=2)[:600], "...")


## 💬 Step 4 — Prompt Templates

In [ ]:
SYSTEM_PROMPT = """You are an expert cybersecurity analyst specializing in URL threat detection.
Your role is to analyze URLs and identify security threats with precision.
Always respond strictly in valid JSON format with no extra text or markdown.""""

# Primary prompt — detailed, structured, high-quality
CLASSIFICATION_PROMPT = """Analyze the following URL for cybersecurity threats:

URL: {url}

Classify it into exactly ONE of these categories:
- "benign"      → Safe, legitimate website
- "phishing"    → Designed to steal credentials or user data
- "malware"     → Distributes or executes malicious software
- "defacement"  → Website that has been hacked or visually defaced

Consider these threat signals when analyzing:
  • Suspicious TLDs (.xyz, .tk, .pw, .ga)
  • IP addresses used instead of domain names
  • Excessive subdomains or long URL paths
  • Brand impersonation (paypal-login.com, amazon-verify.net)
  • Obfuscated or encoded characters (%XX, hex)
  • Known malware distribution patterns

Respond ONLY with valid JSON matching this exact structure (no markdown, no extra text):
{{
    "classification": "benign|phishing|malware|defacement",
    "confidence": 0.85,
    "risk_indicators": ["indicator 1", "indicator 2"],
    "reasoning": "One or two sentences explaining the decision.",
    "severity": "low|medium|high|critical"
}}""""

# Fallback prompt — shorter, cheaper, faster
SIMPLIFIED_PROMPT = """Classify this URL for cybersecurity threats. JSON only.

URL: {url}

Categories: benign, phishing, malware, defacement
Severity: low, medium, high, critical

Respond ONLY with:
{{"classification": "...", "confidence": 0.0, "risk_indicators": ["..."], "reasoning": "...", "severity": "..."}}""""

print("✅ Prompt templates ready")
print(f"   Primary prompt  : {len(CLASSIFICATION_PROMPT)} chars")
print(f"   Fallback prompt : {len(SIMPLIFIED_PROMPT)} chars  ({len(SIMPLIFIED_PROMPT)/len(CLASSIFICATION_PROMPT)*100:.0f}% cost reduction)")


## 🤖 Step 5 — Gemini Classifier

In [ ]:
class GeminiSecurityClassifier:
    """Classifies URLs using Google Gemini 1.5 Flash with structured prompts."""

    def __init__(self, use_simplified_prompt: bool = False, verbose: bool = False):
        self.model = genai.GenerativeModel('gemini-1.5-flash')
        self.use_simplified = use_simplified_prompt
        self.verbose = verbose
        self.call_count  = 0
        self.error_count = 0
        self.parse_errors = 0

    def classify(self, url: str) -> Optional[SecurityClassification]:
        template = SIMPLIFIED_PROMPT if self.use_simplified else CLASSIFICATION_PROMPT
        full_prompt = f"{SYSTEM_PROMPT}\n\n{template.format(url=url)}"

        try:
            response = self.model.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.05,   # Low temp → deterministic outputs
                    max_output_tokens=350,
                )
            )
            self.call_count += 1
            raw = response.text.strip()

            # Strip markdown fences if model wraps in ```json ... ```
            raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()

            data = json.loads(raw)
            result = SecurityClassification(**data)

            if self.verbose:
                print(f"  ✔ [{result.classification.upper():11}] conf={result.confidence:.2f} | {url[:55]}")
            return result

        except json.JSONDecodeError:
            self.parse_errors += 1
            if self.verbose:
                print(f"  ✗ JSON parse error for: {url[:55]}")
            return None
        except Exception as e:
            self.error_count += 1
            if self.verbose:
                print(f"  ✗ API error: {str(e)[:80]}")
            time.sleep(3)
            return None

    @property
    def success_rate(self) -> float:
        total = self.call_count + self.error_count + self.parse_errors
        return self.call_count / total if total > 0 else 0.0

    def __repr__(self):
        return (f"GeminiClassifier(calls={self.call_count}, "
                f"errors={self.error_count}, success={self.success_rate:.1%})")


## 🔍 Step 6 — AI Quality Validator

In [ ]:
class AIQualityValidator:
    """Validates LLM output quality to ensure correctness and safety."""

    SEVERITY_RANK = {"low": 1, "medium": 2, "high": 3, "critical": 4}

    def __init__(self, min_confidence: float = 0.60, min_indicators: int = 1,
                 fallback_threshold: float = 0.55):
        self.min_confidence       = min_confidence
        self.min_indicators       = min_indicators
        self.fallback_threshold   = fallback_threshold
        self.log: List[dict]      = []

    def validate(self, result: SecurityClassification, url: str) -> ValidationResult:
        issues       = []
        quality_score = 1.0

        # ── Check 1: Confidence threshold ─────────────────────────────────
        if result.confidence < self.min_confidence:
            issues.append(f"Low confidence ({result.confidence:.2f} < {self.min_confidence})")
            quality_score -= 0.30

        # ── Check 2: Risk indicators completeness ─────────────────────────
        if len(result.risk_indicators) < self.min_indicators:
            issues.append("Missing risk indicators")
            quality_score -= 0.20

        # ── Check 3: Reasoning meaningfulness ─────────────────────────────
        if len(result.reasoning.strip()) < 15:
            issues.append("Reasoning too brief")
            quality_score -= 0.20

        # ── Check 4: Severity ↔ Classification consistency ────────────────
        if result.classification == "benign" and result.severity in ("high", "critical"):
            issues.append("Inconsistency: benign + high severity")
            quality_score -= 0.25
        if result.classification in ("malware", "phishing") and result.severity == "low":
            issues.append("Inconsistency: threat class + low severity")
            quality_score -= 0.15

        # ── Check 5: Confidence ↔ Severity alignment ──────────────────────
        if result.confidence > 0.90 and result.classification != "benign":
            if result.severity == "low":
                issues.append("High confidence threat but low severity")
                quality_score -= 0.10

        quality_score    = max(0.0, round(quality_score, 3))
        should_fallback  = (quality_score < self.fallback_threshold) or (len(issues) >= 3)

        vr = ValidationResult(
            is_valid       = (len(issues) == 0),
            quality_score  = quality_score,
            issues         = issues,
            should_fallback = should_fallback
        )

        self.log.append({
            "url":              url[:70],
            "classification":   result.classification,
            "confidence":       result.confidence,
            "quality_score":    quality_score,
            "issues":           issues,
            "fallback":         should_fallback,
        })
        return vr

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame(self.log)


## 🪂 Step 7 — HuggingFace Fallback Classifier

In [ ]:
class HuggingFaceFallbackClassifier:
    """
    Cost-sensitive fallback using zero-shot NLI on DistilRoBERTa.
    Activates when Gemini quality score falls below threshold.
    Free, local, no API quota consumed.
    """

    _LABEL_MAP = {
        "safe and legitimate website": "benign",
        "phishing or credential theft": "phishing",
        "malware distribution site":    "malware",
        "defaced or hacked website":    "defacement",
    }
    _SEVERITY  = {"benign": "low", "phishing": "high",
                  "malware": "critical", "defacement": "medium"}

    def __init__(self):
        print("⏳ Loading HuggingFace fallback model (cross-encoder/nli-distilroberta-base)...")
        self._clf = hf_pipeline(
            "zero-shot-classification",
            model="cross-encoder/nli-distilroberta-base",
            device=-1   # CPU
        )
        self.candidate_labels = list(self._LABEL_MAP.keys())
        self.call_count = 0
        print("✅ Fallback model ready!")

    def classify(self, url: str) -> SecurityClassification:
        self.call_count += 1
        out   = self._clf(url, self.candidate_labels)
        label = out["labels"][0]
        score = out["scores"][0]
        cls   = self._LABEL_MAP[label]

        return SecurityClassification(
            classification   = cls,
            confidence       = round(score, 4),
            risk_indicators  = self._heuristic_indicators(url),
            reasoning        = f"Zero-shot NLI classified as '{label}' (score={score:.3f}).",
            severity         = self._SEVERITY[cls],
        )

    def _heuristic_indicators(self, url: str) -> List[str]:
        indicators, u = [], url.lower()
        kw_phishing  = ['login','verify','secure','bank','account','update','confirm']
        kw_malware   = ['download','exe','payload','shell','cmd','run','install']
        for kw in kw_phishing:
            if kw in u: indicators.append(f"keyword:'{kw}'")
        for kw in kw_malware:
            if kw in u: indicators.append(f"keyword:'{kw}'")
        if len(url) > 75:                              indicators.append("long URL")
        if url.count('.') > 4:                         indicators.append("excessive subdomains")
        if re.search(r'\d{1,3}(\.\d{1,3}){3}', url): indicators.append("IP address in URL")
        if '@' in url:                                 indicators.append("@ symbol")
        if re.search(r'%[0-9a-fA-F]{2}', url):        indicators.append("URL encoding")
        return indicators if indicators else ["no obvious indicators"]

# Instantiate once (downloads ~80 MB model)
hf_fallback = HuggingFaceFallbackClassifier()


## 🏗️ Step 8 — Cost-Sensitive Fallback Pipeline

In [ ]:
class SecurityClassificationPipeline:
    """
    4-level cost-sensitive degradation pipeline:
      Level 0 → Gemini primary (full prompt, best quality)
      Level 1 → Gemini simplified (reduced prompt, ~60% cheaper)
      Level 2 → HuggingFace zero-shot (free, local)
      Level 3 → Heuristic rule-based (instant, no model needed)
    """

    def __init__(self, verbose: bool = True):
        self.primary    = GeminiSecurityClassifier(use_simplified_prompt=False, verbose=verbose)
        self.simplified = GeminiSecurityClassifier(use_simplified_prompt=True,  verbose=verbose)
        self.hf         = hf_fallback
        self.validator  = AIQualityValidator(min_confidence=0.65, fallback_threshold=0.55)
        self.verbose    = verbose
        self.results    = []
        self.stats      = {
            "gemini_primary":     0,
            "gemini_simplified":  0,
            "huggingface":        0,
            "heuristic":          0,
            "total_api_errors":   0,
        }

    # ── Level 3: pure rule-based heuristic ───────────────────────────────
    def _heuristic_classify(self, url: str) -> SecurityClassification:
        u   = url.lower()
        cls = "benign"
        sev = "low"
        ind = self.hf._heuristic_indicators(url)

        if re.search(r'\d{1,3}(\.\d{1,3}){3}', url):           cls, sev = "malware",   "critical"
        elif any(k in u for k in ['login','verify','secure']):    cls, sev = "phishing",  "high"
        elif any(k in u for k in ['download','exe','payload']):   cls, sev = "malware",   "high"
        elif url.count('.') > 5:                                   cls, sev = "phishing",  "medium"

        return SecurityClassification(
            classification  = cls,
            confidence      = 0.45,
            risk_indicators = ind,
            reasoning       = "Heuristic rule-based classification (fallback of last resort).",
            severity        = sev,
        )

    def classify_url(self, url: str, true_label: str = None) -> dict:
        entry = dict(url=url[:85], true_label=true_label,
                     predicted_label=None, confidence=0.0,
                     quality_score=0.0, pipeline_level=None,
                     fallback_triggered=False, issues=[])

        def _fill(clf_result, level_name):
            entry["predicted_label"] = clf_result.classification
            entry["confidence"]      = clf_result.confidence
            entry["pipeline_level"]  = level_name
            self.stats[level_name]  += 1

        # ── Level 0: Gemini primary ───────────────────────────────────────
        res = self.primary.classify(url)
        if res:
            vr = self.validator.validate(res, url)
            entry["quality_score"] = vr.quality_score
            entry["issues"]        = vr.issues
            if not vr.should_fallback:
                _fill(res, "gemini_primary")
                self.results.append(entry)
                return entry

        # ── Level 1: Gemini simplified ────────────────────────────────────
        entry["fallback_triggered"] = True
        res2 = self.simplified.classify(url)
        if res2:
            vr2 = self.validator.validate(res2, url)
            entry["quality_score"] = vr2.quality_score
            entry["issues"]        = vr2.issues
            if not vr2.should_fallback:
                _fill(res2, "gemini_simplified")
                self.results.append(entry)
                return entry

        # ── Level 2: HuggingFace fallback ────────────────────────────────
        res3 = self.hf.classify(url)
        vr3  = self.validator.validate(res3, url)
        entry["quality_score"] = vr3.quality_score
        entry["issues"]        = vr3.issues
        if not vr3.should_fallback:
            _fill(res3, "huggingface")
            self.results.append(entry)
            return entry

        # ── Level 3: Heuristic (last resort) ─────────────────────────────
        res4 = self._heuristic_classify(url)
        _fill(res4, "heuristic")
        self.stats["total_api_errors"] += 1
        self.results.append(entry)
        return entry

    def run(self, df: pd.DataFrame, url_col="url", label_col="type",
            max_samples: int = 80, delay: float = 2.0) -> pd.DataFrame:
        """Run the pipeline with rate-limit-safe delays."""
        subset = df.head(max_samples)
        print(f"\n🚀 Security Classification Pipeline  ({len(subset)} URLs)")
        print(f"   Rate-limit delay: {delay}s between Gemini calls")
        print("=" * 60)

        for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Classifying"):
            self.classify_url(str(row[url_col]), str(row[label_col]))
            time.sleep(delay)

        print("\n✅ Pipeline complete!")
        self._print_stats()
        return pd.DataFrame(self.results)

    def _print_stats(self):
        total = sum(v for k, v in self.stats.items() if k != "total_api_errors")
        print("\n📈 Pipeline Level Distribution:")
        bars = {"gemini_primary": "🟢", "gemini_simplified": "🟡",
                "huggingface": "🔴", "heuristic": "⚫"}
        for lvl, icon in bars.items():
            n   = self.stats[lvl]
            pct = n / total * 100 if total else 0
            print(f"  {icon} {lvl:22} {n:3d} ({pct:5.1f}%)")
        print(f"  ⚠️  API errors (heuristic fallback): {self.stats['total_api_errors']}")


## ▶️ Step 9 — Run the Pipeline

> ⏱️ Processing 80 URLs at 2s delay ≈ ~3 minutes  
> Free Gemini tier: **15 requests/min, 1M tokens/day** — we stay well within limits


In [ ]:
pipe = SecurityClassificationPipeline(verbose=True)

results_df = pipe.run(
    sample_df,
    url_col     = "url",
    label_col   = "type",
    max_samples = 80,    # Adjust based on your rate limit quota
    delay       = 2.0,   # Seconds between API calls
)

print(f"\n📋 Results shape: {results_df.shape}")
results_df.head(10)


## 📊 Step 10 — Metrics & False Positive Analysis

In [ ]:
valid = results_df.dropna(subset=['true_label','predicted_label']).copy()
labels_order = ['benign','phishing','malware','defacement']

print("=" * 60)
print("         CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(valid['true_label'], valid['predicted_label'],
                             labels=labels_order, zero_division=0))

# ── False Positive Rate (main project KPI) ────────────────────────────────
benign_mask = valid['true_label'] == 'benign'
fp_mask     = benign_mask & (valid['predicted_label'] != 'benign')

total_benign  = benign_mask.sum()
false_positives = fp_mask.sum()
fpr = false_positives / total_benign if total_benign > 0 else 0

print(f"\n🎯 PROJECT KPIs")
print(f"  False Positive Rate : {fpr:.1%}  (goal: < 20%)")
print(f"  FP count            : {false_positives} / {total_benign} benign URLs misclassified")
print(f"  Macro F1-Score      : {f1_score(valid['true_label'], valid['predicted_label'], average='macro', labels=labels_order, zero_division=0):.3f}")

# ── Fallback trigger rate ─────────────────────────────────────────────────
fb_rate = results_df['fallback_triggered'].mean()
print(f"  Fallback rate       : {fb_rate:.1%}  (lower = better LLM quality)")
print(f"  Avg quality score   : {results_df['quality_score'].mean():.3f} / 1.0")


## 📈 Step 11 — Results Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("LLM Security Classification System — Results Dashboard",
             fontsize=15, fontweight='bold', y=1.01)

COLORS = {'benign':'#2ecc71','phishing':'#e74c3c',
          'malware':'#e67e22','defacement':'#9b59b6'}
LEVEL_COLORS = {'gemini_primary':'#27ae60','gemini_simplified':'#f39c12',
                'huggingface':'#e74c3c','heuristic':'#7f8c8d'}

# ── 1. Confusion Matrix ───────────────────────────────────────────────────
cm = confusion_matrix(valid['true_label'], valid['predicted_label'], labels=labels_order)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=labels_order, yticklabels=labels_order,
            linewidths=0.5, linecolor='white')
axes[0,0].set_title('Confusion Matrix', fontweight='bold')
axes[0,0].set_xlabel('Predicted'); axes[0,0].set_ylabel('Actual')

# ── 2. Pipeline Level Distribution ───────────────────────────────────────
lv_counts = results_df['pipeline_level'].value_counts()
wedge_colors = [LEVEL_COLORS.get(l,'grey') for l in lv_counts.index]
axes[0,1].pie(lv_counts.values, labels=lv_counts.index,
              autopct='%1.1f%%', colors=wedge_colors,
              startangle=140, pctdistance=0.82)
axes[0,1].set_title('Classification by Pipeline Level', fontweight='bold')

# ── 3. Class-wise Accuracy ────────────────────────────────────────────────
class_acc = {}
for cls in labels_order:
    mask     = valid['true_label'] == cls
    acc      = (valid.loc[mask,'predicted_label'] == cls).mean() if mask.sum() else 0
    class_acc[cls] = acc
bars = axes[0,2].bar(class_acc.keys(), class_acc.values(),
                     color=[COLORS[c] for c in class_acc.keys()],
                     edgecolor='white', linewidth=0.8)
axes[0,2].set_ylim(0, 1.15); axes[0,2].set_title('Per-Class Accuracy', fontweight='bold')
axes[0,2].set_ylabel('Accuracy')
for bar, val in zip(bars, class_acc.values()):
    axes[0,2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                   f'{val:.0%}', ha='center', fontsize=10)

# ── 4. Confidence Distribution ────────────────────────────────────────────
for cls in labels_order:
    sub = results_df[results_df['predicted_label'] == cls]['confidence']
    if len(sub): axes[1,0].hist(sub, bins=15, alpha=0.65,
                                 label=cls, color=COLORS[cls])
axes[1,0].axvline(0.65, color='red', linestyle='--', linewidth=1.5, label='Min threshold (0.65)')
axes[1,0].set_title('Confidence Score Distribution', fontweight='bold')
axes[1,0].set_xlabel('Confidence'); axes[1,0].set_ylabel('Count')
axes[1,0].legend(fontsize=8)

# ── 5. Quality Score Distribution ────────────────────────────────────────
axes[1,1].hist(results_df['quality_score'], bins=20, color='steelblue',
               edgecolor='white', linewidth=0.6)
axes[1,1].axvline(0.55, color='orange', linestyle='--', linewidth=1.5,
                  label='Fallback threshold (0.55)')
axes[1,1].axvline(results_df['quality_score'].mean(), color='green',
                  linestyle='-', linewidth=1.5,
                  label=f"Mean ({results_df['quality_score'].mean():.2f})")
axes[1,1].set_title('Validator Quality Scores', fontweight='bold')
axes[1,1].set_xlabel('Quality Score'); axes[1,1].set_ylabel('Count')
axes[1,1].legend(fontsize=8)

# ── 6. False Positive Summary ─────────────────────────────────────────────
fp_summary = {
    'True Positives\n(threats detected)':
        ((valid['true_label']!='benign') & (valid['predicted_label']!='benign')).sum(),
    'True Negatives\n(benign correct)':
        ((valid['true_label']=='benign') & (valid['predicted_label']=='benign')).sum(),
    'False Positives\n(benign→threat)':
        ((valid['true_label']=='benign') & (valid['predicted_label']!='benign')).sum(),
    'False Negatives\n(threat missed)':
        ((valid['true_label']!='benign') & (valid['predicted_label']=='benign')).sum(),
}
fp_colors = ['#2ecc71','#3498db','#e74c3c','#e67e22']
bars2 = axes[1,2].bar(range(4), fp_summary.values(), color=fp_colors, edgecolor='white')
axes[1,2].set_xticks(range(4))
axes[1,2].set_xticklabels(fp_summary.keys(), fontsize=7.5)
axes[1,2].set_title('Prediction Outcome Summary', fontweight='bold')
axes[1,2].set_ylabel('Count')
for bar, val in zip(bars2, fp_summary.values()):
    axes[1,2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                   str(val), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("security_classification_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Dashboard saved as security_classification_dashboard.png")


## 🔍 Step 12 — Quality Validator Analysis

In [ ]:
val_df = pipe.validator.summary()

print("📋 Validator Log Sample:")
display(val_df.tail(10))

print(f"\n📊 Validation Statistics:")
print(f"  Total validations   : {len(val_df)}")
print(f"  Fallbacks triggered : {val_df['fallback'].sum()} ({val_df['fallback'].mean():.1%})")
print(f"  Avg quality score   : {val_df['quality_score'].mean():.3f}")
print(f"  High quality (≥0.8) : {(val_df['quality_score'] >= 0.8).sum()} ({(val_df['quality_score'] >= 0.8).mean():.1%})")

# Most common quality issues
all_issues = [issue for issues_list in val_df['issues'] for issue in issues_list]
if all_issues:
    from collections import Counter
    issue_counts = Counter(all_issues).most_common(5)
    print("\n⚠️  Top Quality Issues:")
    for issue, count in issue_counts:
        print(f"  {count:3d}x  {issue}")
else:
    print("\n✅ No quality issues detected!")


## 💾 Step 13 — Export Results

In [ ]:
# Save classification results
results_df.to_csv("security_classification_results.csv", index=False)

# Save validation log
val_df.to_csv("quality_validation_log.csv", index=False)

# Final summary report
summary = {
    "total_urls_classified": len(results_df),
    "pipeline_distribution": pipe.stats,
    "false_positive_rate": float(fpr),
    "macro_f1": float(f1_score(valid['true_label'], valid['predicted_label'],
                               average='macro', labels=labels_order, zero_division=0)),
    "avg_quality_score": float(results_df['quality_score'].mean()),
    "fallback_rate": float(results_df['fallback_triggered'].mean()),
    "gemini_primary_calls": pipe.primary.call_count,
    "gemini_simplified_calls": pipe.simplified.call_count,
    "huggingface_calls": pipe.hf.call_count,
}

with open("pipeline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Files saved:")
print("   📄 security_classification_results.csv")
print("   📄 quality_validation_log.csv")
print("   📄 pipeline_summary.json")
print("   📊 security_classification_dashboard.png")
print()
print("📌 Final Summary:")
for k, v in summary.items():
    if isinstance(v, float): print(f"   {k:35} {v:.3f}")
    elif isinstance(v, dict): pass
    else: print(f"   {k:35} {v}")
